# Correlação, forma e causalidade

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_06/02_correlacao_variaveis_numericas.ipynb)

Correlação de Pearson resume associação **linear**; Spearman calcula Pearson
sobre postos e resume associação **monotônica**. Ambas variam de -1 a 1, mas um
coeficiente isolado pode ocultar curvas, grupos, extremos e variáveis de confusão.

$$r=\frac{\sum_i(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_i(x_i-\bar{x})^2\sum_i(y_i-\bar{y})^2}}$$

## 1. Ver a forma antes de resumir

A célula seguinte lê `registros_associacao.csv`, usa cada documento como unidade
e compara extensão com referências institucionais. O gráfico vem antes dos
coeficientes porque curvas, grupos e valores extremos podem produzir leituras
muito diferentes de um mesmo número.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_06'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

dados = pd.read_csv("dados/registros_associacao.csv")
x = dados["extensao_palavras"]
y = dados["referencias_instituicoes"]
pearson = x.corr(y)
spearman = x.rank(method="average").corr(y.rank(method="average"))
print(f"Pearson: {pearson:.3f}")
print(f"Spearman: {spearman:.3f}")

cores = dados["local"].map({"Capital": "#a44a3f", "Interior": "#4c768b"})
plt.figure(figsize=(7, 4.5))
plt.scatter(x, y, c=cores, alpha=.75)
plt.xlabel("Extensão do documento (palavras)")
plt.ylabel("Referências a instituições")
plt.title("Forma e grupos antes do coeficiente — dados fictícios")
plt.tight_layout(); plt.show()

## 2. Sensibilidade, estratificação e confundimento

Um coeficiente agregado combina relações internas e diferenças entre grupos.
Primeiro recalcularemos **extensão × referências institucionais** por local. Depois
usaremos um segundo par — **extensão × circulação estimada** — para demonstrar uma
análise de sensibilidade com e sem o maior valor de circulação. Não compare os
coeficientes dos dois pares como se fossem versões da mesma pergunta. A finalidade
é entender de que decisões cada resultado depende.

In [ ]:
por_local = dados.groupby("local").apply(
    lambda g: pd.Series({
        "n": len(g),
        "Pearson": g["extensao_palavras"].corr(g["referencias_instituicoes"]),
        "Spearman": g["extensao_palavras"].rank().corr(g["referencias_instituicoes"].rank()),
    }), include_groups=False
)
pearson_circulacao = dados["extensao_palavras"].corr(dados["circulacao_estimada"])
sem_extremo = dados.drop(dados["circulacao_estimada"].idxmax())
sensibilidade = pd.Series({
    "extensão × circulação — conjunto completo": pearson_circulacao,
    "extensão × circulação — sem maior circulação": sem_extremo["extensao_palavras"].corr(sem_extremo["circulacao_estimada"]),
})
display(por_local.round(3)); display(sensibilidade.round(3).to_frame("coeficiente"))

![Associação observada pode refletir direção causal, causa reversa, fator comum, seleção ou coincidência.](imagens/02_correlacao_causalidade.svg)

Correlação não estabelece temporalidade, mecanismo, ausência de confundimento ou
efeito de intervenção. A expressão “X causa Y” exige desenho e evidências além do
coeficiente. Correlações espúrias também podem surgir de tendências temporais,
testes múltiplos ou seleção posterior das variáveis.

## U06-A03 — Atividade integrada — correlação com auditoria

Analise `qualidade_ocr_pct` e `ano`, ou `circulacao_estimada` e
`extensao_palavras`. Apresente gráfico, Pearson, Spearman, estratificação por um
grupo, sensibilidade a um extremo e três explicações rivais.

**Resposta:** Escreva aqui.

## Passagem

Até aqui a relação existia entre colunas. O Notebook 03 mostra como documentos
podem ser transformados em relações entre entidades — uma mudança de ontologia,
não apenas de formato.

**Leitura:** Pearson (1896), Spearman (1904) e Pearl (2009), listados em
`referencias.md`.